# LightGBM LambdaRank Model

Ranking model using LightGBM's `lambdarank` objective, which optimises NDCG directly.

In [1]:
import sys
from pathlib import Path

cwd = Path(".").resolve()
REPO_ROOT = cwd if (cwd / ".git").exists() else cwd.parent
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import lightgbm as lgb

from src.utils import load_train_cached_sample, train_val_split, compute_ndcg_fast, make_submission
from src.features import build_features

In [2]:
SAMPLE_FRAC = 0.15
REFRESH_CACHE = False

df = load_train_cached_sample(sample_frac=SAMPLE_FRAC, refresh=REFRESH_CACHE)
n_cols_before = df.shape[1]
df = build_features(df)
print(f"Loaded {len(df):,} rows, {df['srch_id'].nunique():,} searches")
print(f"Columns: {n_cols_before} raw + {df.shape[1] - n_cols_before} engineered = {df.shape[1]} total")

nan_pct = df.isnull().mean()
print(f"\nColumns with NaNs:")
print((nan_pct[nan_pct > 0] * 100).round(1).to_string())

Loaded 742,438 rows, 29,969 searches
Columns: 54 raw + 40 engineered = 94 total

Columns with NaNs:
visitor_hist_starrating             95.0
visitor_hist_adr_usd                95.0
prop_review_score                    0.1
prop_location_score2                22.2
srch_query_affinity_score           93.5
orig_destination_distance           33.0
comp1_rate                          97.5
comp1_inv                           97.3
comp1_rate_percent_diff             98.1
comp2_rate                          59.7
comp2_inv                           57.6
comp2_rate_percent_diff             88.9
comp3_rate                          69.5
comp3_inv                           67.3
comp3_rate_percent_diff             90.6
comp4_rate                          93.8
comp4_inv                           93.1
comp4_rate_percent_diff             97.4
comp5_rate                          55.6
comp5_inv                           52.8
comp5_rate_percent_diff             83.3
comp6_rate                          95.

In [3]:
train_df, val_df = train_val_split(df)

def make_relevance(frame):
    """booking=5, click-only=1, nothing=0"""
    return (frame["booking_bool"] * 5 +
            (frame["click_bool"] - frame["booking_bool"]).clip(lower=0)).values

y_train = make_relevance(train_df)
y_val = make_relevance(val_df)

DROP_COLS = [
    "srch_id", "date_time", "position", "click_bool", "booking_bool",
    "gross_bookings_usd", "random_bool",
]
COMP_RAW = [f"comp{i}_{s}" for i in range(1, 9) for s in ("rate", "inv", "rate_percent_diff")]
drop = set(DROP_COLS + COMP_RAW)
feature_cols = [c for c in train_df.columns if c not in drop]

print(f"Train: {train_df.shape[0]:,}  Val: {val_df.shape[0]:,}")
print(f"Features: {len(feature_cols)}")

X_train = train_df[feature_cols]
X_val = val_df[feature_cols]

group_train = train_df.groupby("srch_id").size().values
group_val = val_df.groupby("srch_id").size().values

Train: 593,717  Val: 148,721
Features: 63


In [4]:
params = {
    "objective": "lambdarank",
    "metric": "ndcg",
    "ndcg_eval_at": [5],
    "num_leaves": 127,
    "learning_rate": 0.05,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "feature_pre_filter": False,
    "min_child_samples": 50,
    "num_threads": -1,
    "verbose": -1,
    "seed": 42,
}

train_set = lgb.Dataset(X_train, label=y_train, group=group_train)
val_set = lgb.Dataset(X_val, label=y_val, group=group_val, reference=train_set)

model = lgb.train(
    params,
    train_set,
    valid_sets=[val_set],
    num_boost_round=2000,
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)],
)

Training until validation scores don't improve for 50 rounds


[100]	valid_0's ndcg@5: 0.370875


Early stopping, best iteration is:
[89]	valid_0's ndcg@5: 0.371429


In [5]:
val_df_eval = val_df.copy()
val_df_eval["pred_score"] = model.predict(X_val)

baseline_ndcg = compute_ndcg_fast(val_df_eval, "prop_location_score1")
model_ndcg = compute_ndcg_fast(val_df_eval, "pred_score")
print(f"Baseline NDCG@5 (location_score1): {baseline_ndcg:.4f}")
print(f"LambdaRank NDCG@5:                 {model_ndcg:.4f}")
print(f"Improvement:                        +{model_ndcg - baseline_ndcg:.4f}")

Baseline NDCG@5 (location_score1): 0.1755
LambdaRank NDCG@5:                 0.3719
Improvement:                        +0.1964


In [6]:
importance = pd.DataFrame({
    "feature": feature_cols,
    "gain": model.feature_importance(importance_type="gain"),
}).sort_values("gain", ascending=False)

print("Top 20 features by gain:")
print(importance.head(20).to_string(index=False))

Top 20 features by gain:
                         feature         gain
            prop_location_score2 36245.434121
       prop_location_score2_rank 27285.303571
             price_vs_historical 17443.677518
                  price_usd_rank 15071.526513
       prop_starrating_diff_mean 14703.368828
            prop_starrating_rank 14030.545615
           price_usd_diff_median 13458.274597
                         prop_id 11891.704497
        prop_location_score2_max 11850.468101
             price_usd_diff_mean 10329.697710
            prop_location_score1  9700.181918
        prop_location_score2_std  9526.929857
          prop_review_score_rank  9442.549290
  prop_location_score2_diff_mean  9246.610494
     prop_review_score_diff_mean  9240.006730
prop_location_score2_diff_median  8874.195014
                             ump  8188.367272
                price_per_person  7525.716590
                         per_fee  7315.791409
prop_location_score1_diff_median  7075.633300


## Hyperparameter Search

Grid search over key parameters. Evaluate each on validation NDCG@5.

In [7]:
from itertools import product

search_space = {
    "num_leaves": [63, 127, 255],
    "learning_rate": [0.01, 0.05, 0.1],
    "min_child_samples": [20, 50, 100],
}

results = []
base_params = {
    "objective": "lambdarank",
    "metric": "ndcg",
    "ndcg_eval_at": [5],
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "feature_pre_filter": False,
    "num_threads": -1,
    "verbose": -1,
    "seed": 42,
}

keys = list(search_space.keys())
for combo in product(*search_space.values()):
    trial_params = {**base_params, **dict(zip(keys, combo))}

    train_set_hp = lgb.Dataset(X_train, label=y_train, group=group_train)
    val_set_hp = lgb.Dataset(X_val, label=y_val, group=group_val, reference=train_set_hp)

    m = lgb.train(
        trial_params,
        train_set_hp,
        valid_sets=[val_set_hp],
        num_boost_round=2000,
        callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)],
    )

    preds = m.predict(X_val)
    val_df_eval["pred_score"] = preds
    score = compute_ndcg_fast(val_df_eval, "pred_score")

    row = dict(zip(keys, combo))
    row["ndcg5"] = score
    row["n_trees"] = m.best_iteration
    results.append(row)
    print(f"  leaves={row['num_leaves']:>3}  lr={row['learning_rate']:.2f}  "
          f"min_child={row['min_child_samples']:>3}  -> NDCG@5={score:.4f}  trees={m.best_iteration}")

results_df = pd.DataFrame(results).sort_values("ndcg5", ascending=False)
print("\nTop 5 configurations:")
print(results_df.head().to_string(index=False))

  leaves= 63  lr=0.01  min_child= 20  -> NDCG@5=0.3759  trees=297


  leaves= 63  lr=0.01  min_child= 50  -> NDCG@5=0.3772  trees=376


  leaves= 63  lr=0.01  min_child=100  -> NDCG@5=0.3820  trees=818


  leaves= 63  lr=0.05  min_child= 20  -> NDCG@5=0.3761  trees=117


  leaves= 63  lr=0.05  min_child= 50  -> NDCG@5=0.3756  trees=76


...

  leaves=255  lr=0.05  min_child= 50  -> NDCG@5=0.3696  trees=81


  leaves=255  lr=0.05  min_child=100  -> NDCG@5=0.3689  trees=146


  leaves=255  lr=0.10  min_child= 20  -> NDCG@5=0.3585  trees=22


  leaves=255  lr=0.10  min_child= 50  -> NDCG@5=0.3553  trees=39


  leaves=255  lr=0.10  min_child=100  -> NDCG@5=0.3592  trees=60

Top 5 configurations:
 num_leaves  learning_rate  min_child_samples    ndcg5  n_trees
         63           0.01                100 0.382026      818
        127           0.01                 20 0.379479      370
        255           0.01                 50 0.377266      350
         63           0.01                 50 0.377231      376
        127           0.01                100 0.376762      339


## Retrain with Best Params

Use the best configuration from the grid search, then save the model.

In [8]:
best = results_df.iloc[0]
print(f"Best config: leaves={int(best.num_leaves)}, lr={best.learning_rate}, "
      f"min_child={int(best.min_child_samples)}, NDCG@5={best.ndcg5:.4f}")

best_params = {
    **base_params,
    "num_leaves": int(best.num_leaves),
    "learning_rate": best.learning_rate,
    "min_child_samples": int(best.min_child_samples),
}

train_set_best = lgb.Dataset(X_train, label=y_train, group=group_train)
val_set_best = lgb.Dataset(X_val, label=y_val, group=group_val, reference=train_set_best)

best_model = lgb.train(
    best_params,
    train_set_best,
    valid_sets=[val_set_best],
    num_boost_round=2000,
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)],
)

val_df_eval["pred_score"] = best_model.predict(X_val)
final_ndcg = compute_ndcg_fast(val_df_eval, "pred_score")
print(f"\nFinal NDCG@5: {final_ndcg:.4f}")

models_dir = REPO_ROOT / "models"
models_dir.mkdir(exist_ok=True)
best_model.save_model(str(models_dir / "lambdarank_best.txt"))

Best config: leaves=63, lr=0.01, min_child=100, NDCG@5=0.3820


Training until validation scores don't improve for 50 rounds


[100]	valid_0's ndcg@5: 0.36922


[200]	valid_0's ndcg@5: 0.373143


[300]	valid_0's ndcg@5: 0.374844


...

[600]	valid_0's ndcg@5: 0.378619


[700]	valid_0's ndcg@5: 0.379113


[800]	valid_0's ndcg@5: 0.380031


Early stopping, best iteration is:
[818]	valid_0's ndcg@5: 0.381413



Final NDCG@5: 0.3820


In [ ]:
import json

out_dir = REPO_ROOT / "outputs"
out_dir.mkdir(exist_ok=True)

existing = sorted(out_dir.glob("lambdarank_run*.json"))
run_num = len(existing) + 1

results_out = {
    "model": "lambdarank",
    "run": run_num,
    "sample_frac": SAMPLE_FRAC,
    "n_rows": len(df),
    "n_searches": int(df["srch_id"].nunique()),
    "n_train": len(train_df),
    "n_val": len(val_df),
    "n_features": len(feature_cols),
    "n_trees": int(best_model.best_iteration),
    "best_params": {k: v for k, v in best_params.items() if k not in ("verbose", "num_threads", "seed")},
    "baseline_ndcg5": round(baseline_ndcg, 4),
    "model_ndcg5": round(final_ndcg, 4),
    "top_features": {row["feature"]: round(row["gain"], 1) for _, row in importance.head(15).iterrows()},
}

filename = f"lambdarank_run{run_num}.json"
with open(out_dir / filename, "w") as f:
    json.dump(results_out, f, indent=2)

print(f"Results saved to outputs/{filename}")